In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.optim as optim 
import pandas 
import torchvision.transforms as transforms

In [2]:
import os
data_path=r"C:\Users\shaik zabiulla\Downloads\archive (7)"
print(os.listdir(data_path))

['test', 'train']


In [3]:
print(os.listdir(data_path + r"\train"))

['cats', 'dogs']


In [4]:
transform=transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5,0.5,0.5],
        std=[0.5,0.5,0.5]

    )
])

In [5]:
from torchvision import datasets

In [6]:
train_dataset=datasets.ImageFolder(r"C:\Users\shaik zabiulla\Downloads\archive (7)\train",transform=transform)
test_datset=datasets.ImageFolder(r"C:\Users\shaik zabiulla\Downloads\archive (7)\test",transform=transform)
print(train_dataset.classes)
print(train_dataset)

['cats', 'dogs']
Dataset ImageFolder
    Number of datapoints: 20000
    Root location: C:\Users\shaik zabiulla\Downloads\archive (7)\train
    StandardTransform
Transform: Compose(
               Resize(size=(128, 128), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
               Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
           )


In [7]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_datset,batch_size=32)

In [8]:
class catvsdog(nn.Module):
    def __init__(self):
        super().__init__()
        self.convo_layer=nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2,2)

        )
        self.fc_layer=nn.Sequential(
            nn.Linear(128*16*16,128),
            nn.ReLU(),
            nn.Linear(128,2)
        )
    def forward(self,x):
        x=self.convo_layer(x)
        x=x.view(x.size(0),-1)
        x=self.fc_layer(x)
        return x
        

        

In [9]:
model=catvsdog()
optimizer=optim.Adam(model.parameters(),lr=0.001)
criteria=nn.CrossEntropyLoss()


In [10]:
img,label=train_dataset[0]
print(type(img))
print(img.shape)
print(label)

<class 'torch.Tensor'>
torch.Size([3, 128, 128])
0


In [11]:
epochs=10
for epoch in range (epochs ):
    model.train()
    current_loss=0
    for images,labels in train_loader:
        optimizer.zero_grad()
        pred=model(images)
        loss=criteria(pred,labels)
        loss.backward()
        optimizer.step()

        current_loss +=loss.item()
    train_loss=current_loss/len(train_loader)
    if (epoch +1) % 2==0 or epoch ==1:
        print(f"for epoch {epoch +1}/{epochs} training loss is {train_loss}")

        


        
   


for epoch 2/10 training loss is 0.4638711709499359
for epoch 4/10 training loss is 0.35052841901779175
for epoch 6/10 training loss is 0.2937742383599281
for epoch 8/10 training loss is 0.233308660030365
for epoch 10/10 training loss is 0.18599439436793327


In [16]:
model.eval()
total=0
correct=0
total_loss=0
for image,label in test_loader:
    with torch.no_grad():
        pred=(model(image))
        loss=criteria(pred,label)
        total_loss += loss.item()

        predicted=torch.argmax(pred,dim=1)
        correct+= (predicted==label.squeeze()).sum().item()
        total+=label.size(0)
accuracy=correct/total*100
print(f"accuracy= {accuracy}")

    
    




accuracy= 85.9
